# Script 17 — 3D Pose Visualization
**Pan-coronavirus RTC Inhibitor Discovery** | GIGA-VIN Lab, ULiège | 2026

## Cell 1 — Imports

In [13]:
import py3Dmol
from pathlib import Path
import numpy as np
from IPython.display import display, Markdown

## Cell 2 — Define Paths

In [14]:
BASE   = Path('.')
POSES  = BASE / '04-hits/poses'
RECEPT = BASE / '03-virtual-screening'

TARGETS = {
    'NSP12-NSP7': RECEPT / 'NSP12-NSP7_3/receptor_NSP12-NSP7_3.pdb',
    'NSP9-NSP12': RECEPT / 'NSP9-NSP12_5/receptor_NSP9-NSP12_5.pdb',
    'NSP12-NSP8': RECEPT / 'NSP12-NSP8_4/receptor_NSP12-NSP8_4.pdb',
}

## Cell 3 — Helper Functions

In [15]:
def pdbqt_to_pdb_str(pdbqt_path):
    
    lines = []
    in_m1 = False
    
    with open(pdbqt_path) as f:
        for line in f:
            
            if line.startswith('MODEL        1'):
                in_m1 = True
                continue
                
            if line.startswith('ENDMDL') and in_m1:
                break
                
            if in_m1 and line.startswith(('ATOM','HETATM')):
                lines.append(line[:66].rstrip())
                
    return '\n'.join(lines)


def make_complex(rec_pdb, lig_pdbqt):
    
    with open(rec_pdb) as f:
        rec = [l.rstrip() for l in f if l.startswith(('ATOM','HETATM','TER'))]

    lig = pdbqt_to_pdb_str(lig_pdbqt).split('\n')

    lig_out = []

    for line in lig:
        
        if line.startswith(('ATOM','HETATM')):
            line = 'HETATM' + line[6:17] + 'LIG Z   1' + line[26:66]
            
        lig_out.append(line)

    return '\n'.join(rec + ['TER'] + lig_out + ['END'])


def get_center(pdbqt_path):
    
    coords = []
    in_m1 = False
    
    with open(pdbqt_path) as f:
        for line in f:
            
            if line.startswith('MODEL        1'):
                in_m1 = True
                
            if line.startswith('ENDMDL') and in_m1:
                break
                
            if in_m1 and line.startswith(('ATOM','HETATM')):
                
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass

    if coords:
        return np.mean(coords, axis=0)
    else:
        return np.zeros(3)

## Cell 4 — Lead Compounds

In [16]:
LEADS = [

    ('351017',  'NSP12-NSP7', -8.153, 'Scaffold A — Lead #1'),
    ('351017',  'NSP9-NSP12', -8.132, 'Scaffold A — Lead #1'),
    ('351017',  'NSP12-NSP8', -7.759, 'Scaffold A — Lead #1'),

    ('5024943', 'NSP12-NSP7', -8.225, 'Scaffold B — Lead #5'),
    ('5024943', 'NSP9-NSP12', -7.261, 'Scaffold B — Lead #5'),

]

## Cell 5 — Visualization Function

In [17]:
def show_complex(rec_path, lig_path):

complex_pdb = make_complex(rec_path, lig_path)

center = get_center(lig_path)

view = py3Dmol.view(width=800, height=500)

view.addModel(complex_pdb, 'pdb')

# receptor
view.setStyle(
    {'chain':{'$ne':'Z'}},
    {'cartoon':{'color':'lightgrey','opacity':0.9}}
)

# ligand
view.setStyle(
    {'chain':'Z'},
    {'stick':{'colorscheme':'greenCarbon','radius':0.25}}
)

# binding surface
view.addSurface(
    py3Dmol.SAS,
    {'opacity':0.08,'color':'lightblue'},
    {'chain':{'$ne':'Z'},'within':{'distance':8,'sel':{'chain':'Z'}}}
)

# ligand centroid
view.addSphere({
    'center':{'x':center[0],'y':center[1],'z':center[2]},
    'radius':1.5,
    'color':'yellow',
    'opacity':0.4
})

view.zoomTo({'chain':'Z'})
view.show()

IndentationError: expected an indented block after function definition on line 1 (3587303179.py, line 3)

## Cell 6 — Display All Docking Poses

In [18]:
for zinc_id, target, score, label in LEADS:

    lig_path = POSES / target / f'{zinc_id}_out.pdbqt'
    rec_path = TARGETS[target]

    if not lig_path.exists():
        print("Missing:", lig_path)
        continue

    display(Markdown(
        f"### ZINC{zinc_id} · {target} · {score:.3f} kcal/mol  \n**{label}**"
    ))

    show_complex(rec_path, lig_path)

Missing: 04-hits/poses/NSP12-NSP7/351017_out.pdbqt
Missing: 04-hits/poses/NSP9-NSP12/351017_out.pdbqt
Missing: 04-hits/poses/NSP12-NSP8/351017_out.pdbqt
Missing: 04-hits/poses/NSP12-NSP7/5024943_out.pdbqt
Missing: 04-hits/poses/NSP9-NSP12/5024943_out.pdbqt
